<a href="https://colab.research.google.com/github/youjung1111/face-aging-morphing/blob/main/gradio_%EC%8B%9C%EB%8F%84demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Gradio + PIL + 경로/저장 등을 위한 필요한 기본 모듈 로딩
import os
from PIL import Image
import shutil
from pathlib import Path

# 함수 뼈대 생성 (내용은 이후 채울 예정)
def run_pipeline(child_img, mom_img, dad_img, name, age_then, age_now, desc, phone):
    # 1. 디렉토리 생성
    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    # 2. 이미지 저장
    child_path = "input/child.jpg"
    mom_path = "input/mom.jpg"
    dad_path = "input/dad.jpg"

    child_img.convert("RGB").resize((256, 256)).save(child_path)
    mom_img.convert("RGB").resize((256, 256)).save(mom_path)
    dad_img.convert("RGB").resize((256, 256)).save(dad_path)

    # 이후: 모델 클론/실행 및 포스터 생성은 shell 또는 subprocess로 실행 예정
    return Image.new("RGB", (512, 512), color="white")  # 테스트용 빈 이미지 반환

# UI 테스트용 기본 Gradio 구성 미리 정의 (전체 연결은 이후 작업)
import gradio as gr

demo = gr.Interface(
    fn=run_pipeline,
    inputs=[
        gr.Image(type="pil", label="👶 아동 사진"),
        gr.Image(type="pil", label="👩 어머니 사진"),
        gr.Image(type="pil", label="👨 아버지 사진"),
        gr.Textbox(label="1) 아동 이름"),
        gr.Textbox(label="2) 당시 나이"),
        gr.Textbox(label="3) 현재 나이"),
        gr.Textbox(label="4) 아동의 특징"),
        gr.Textbox(label="5) 연락처"),
    ],
    outputs=gr.Image(type="pil", label="🖼️ 생성된 포스터"),
    title="실종 아동 예측 시스템",
    description="아동 사진 및 부모 사진을 기반으로, 예측 얼굴과 정보를 포함한 포스터를 생성합니다."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6b61d5bf3999a60fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# 📦 설치 및 폴더 준비
!pip install gradio --quiet
import os
import subprocess
from PIL import Image, ImageDraw

os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)

# 🖼️ 포스터 생성 함수
def generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone):
    poster = Image.new("RGB", (600, 900), color="white")
    draw = ImageDraw.Draw(poster)

    aged_img = Image.open(aged_img_path).resize((256, 256))
    blend_img = Image.open(blended_img_path).resize((256, 256))

    poster.paste(aged_img, (50, 30))
    poster.paste(blend_img, (300, 30))

    y = 320
    spacing = 40
    draw.text((50, y), f"이름: {name}", fill="black")
    draw.text((50, y+1*spacing), f"당시 나이: {age_then}", fill="black")
    draw.text((50, y+2*spacing), f"현재 나이: {age_now}", fill="black")
    draw.text((50, y+3*spacing), f"특징: {desc}", fill="black")
    draw.text((50, y+4*spacing), f"연락처: {phone}", fill="black")
    return poster

# 🧠 전체 파이프라인 함수
def run_pipeline(child_img, mom_img, dad_img, name, age_then, age_now, desc, phone):
    # 1. 이미지 저장
    child_path = "/content/face-aging-morphing/aging_model/input.jpg"
    mom_path = "/content/input/test1_mom.jpg"
    dad_path = "/content/input/test1_dad.jpg"

    child_img.convert("RGB").resize((256, 256)).save(child_path)
    mom_img.convert("RGB").resize((256, 256)).save(mom_path)
    dad_img.convert("RGB").resize((256, 256)).save(dad_path)

    # 2. Fast-AgingGAN 실행
    os.chdir("/content/face-aging-morphing/aging_model")
    subprocess.run(["python", "infer.py", "--image_dir", "."], check=True)

    aged_img_path = "results/input_aged.jpg"

    # 3. MorphGAN 실행 준비
    os.chdir("/content/face-aging-morphing/morphing_model")
    os.makedirs("../datasets/mom_dad", exist_ok=True)
    subprocess.run(["cp", mom_path, "../datasets/mom_dad/test1_A.jpg"])
    subprocess.run(["cp", dad_path, "../datasets/mom_dad/test1_B.jpg"])
    with open("../datasets/mom_dad/pairs.txt", "w") as f:
        f.write("test1_A.jpg test1_B.jpg")

    subprocess.run(["chmod", "+x", "runt.sh"])
    subprocess.run(["./runt.sh", "-n", "mom_dad", "-d", "../datasets/mom_dad", "-v", "mom_dad", "-s", "256"], check=True)

    blended_img_path = "results/mom_dad/test_latest/images/test1_A_intr_1.00.png"

    # 4. 포스터 생성
    poster = generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone)
    out_path = f"/content/output/poster_{name.replace(' ', '_')}.jpg"
    poster.save(out_path)
    return poster

# 🌐 Gradio UI 연결
import gradio as gr

demo = gr.Interface(
    fn=run_pipeline,
    inputs=[
        gr.Image(type="pil", label="👶 아동 사진"),
        gr.Image(type="pil", label="👩 어머니 사진"),
        gr.Image(type="pil", label="👨 아버지 사진"),
        gr.Textbox(label="1) 아동 이름"),
        gr.Textbox(label="2) 당시 나이"),
        gr.Textbox(label="3) 현재 나이"),
        gr.Textbox(label="4) 아동의 특징"),
        gr.Textbox(label="5) 연락처"),
    ],
    outputs=gr.Image(type="pil", label="🖼️ 생성된 포스터"),
    title="실종 아동 예측 시스템 (AI 기반)",
    description="아동과 부모 사진을 입력하면 AI로 예측된 얼굴을 기반으로 포스터를 생성합니다."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c418085b0c177f2f0d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
# 완성된 run_pipeline() + Gradio UI를 한 셀로 실행할 수 있도록 구성
# 모델 실행 명령어와 이미지 경로를 기반으로 구조화된 셀 코드 생성

gradio_pipeline_code = """
# 📦 설치 및 폴더 준비
!pip install gradio --quiet
import os
import subprocess
from PIL import Image, ImageDraw

os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)

# 🖼️ 포스터 생성 함수
def generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone):
    poster = Image.new("RGB", (600, 900), color="white")
    draw = ImageDraw.Draw(poster)

    aged_img = Image.open(aged_img_path).resize((256, 256))
    blend_img = Image.open(blended_img_path).resize((256, 256))

    poster.paste(aged_img, (50, 30))
    poster.paste(blend_img, (300, 30))

    y = 320
    spacing = 40
    draw.text((50, y), f"이름: {name}", fill="black")
    draw.text((50, y+1*spacing), f"당시 나이: {age_then}", fill="black")
    draw.text((50, y+2*spacing), f"현재 나이: {age_now}", fill="black")
    draw.text((50, y+3*spacing), f"특징: {desc}", fill="black")
    draw.text((50, y+4*spacing), f"연락처: {phone}", fill="black")
    return poster

# 🧠 전체 파이프라인 함수
def run_pipeline(child_img, mom_img, dad_img, name, age_then, age_now, desc, phone):
    # 1. 이미지 저장
    child_path = "/content/face-aging-morphing/aging_model/input.jpg"
    mom_path = "/content/input/test1_mom.jpg"
    dad_path = "/content/input/test1_dad.jpg"

    child_img.convert("RGB").resize((256, 256)).save(child_path)
    mom_img.convert("RGB").resize((256, 256)).save(mom_path)
    dad_img.convert("RGB").resize((256, 256)).save(dad_path)

    # 2. Fast-AgingGAN 실행
    os.chdir("/content/face-aging-morphing/aging_model")
    subprocess.run(["python", "infer.py", "--image_dir", "."], check=True)

    aged_img_path = "results/input_aged.jpg"

    # 3. MorphGAN 실행 준비
    os.chdir("/content/face-aging-morphing/morphing_model")
    os.makedirs("../datasets/mom_dad", exist_ok=True)
    subprocess.run(["cp", mom_path, "../datasets/mom_dad/test1_A.jpg"])
    subprocess.run(["cp", dad_path, "../datasets/mom_dad/test1_B.jpg"])
    with open("../datasets/mom_dad/pairs.txt", "w") as f:
        f.write("test1_A.jpg test1_B.jpg")

    subprocess.run(["chmod", "+x", "runt.sh"])
    subprocess.run(["./runt.sh", "-n", "mom_dad", "-d", "../datasets/mom_dad", "-v", "mom_dad", "-s", "256"], check=True)

    blended_img_path = "results/mom_dad/test_latest/images/test1_A_intr_1.00.png"

    # 4. 포스터 생성
    poster = generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone)
    out_path = f"/content/output/poster_{name.replace(' ', '_')}.jpg"
    poster.save(out_path)
    return poster

# 🌐 Gradio UI 연결
import gradio as gr

demo = gr.Interface(
    fn=run_pipeline,
    inputs=[
        gr.Image(type="pil", label="👶 아동 사진"),
        gr.Image(type="pil", label="👩 어머니 사진"),
        gr.Image(type="pil", label="👨 아버지 사진"),
        gr.Textbox(label="1) 아동 이름"),
        gr.Textbox(label="2) 당시 나이"),
        gr.Textbox(label="3) 현재 나이"),
        gr.Textbox(label="4) 아동의 특징"),
        gr.Textbox(label="5) 연락처"),
    ],
    outputs=gr.Image(type="pil", label="🖼️ 생성된 포스터"),
    title="실종 아동 예측 시스템 (AI 기반)",
    description="아동과 부모 사진을 입력하면 AI로 예측된 얼굴을 기반으로 포스터를 생성합니다."
)

demo.launch(share=True)
"""

# 저장
path = Path("/mnt/data/final_gradio_pipeline.ipynb")
path.write_text(gradio_pipeline_code)
path.name


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/final_gradio_pipeline.ipynb'

In [10]:
# 확인 1: 노화 이미지 생성 여부
!ls /content/face-aging-morphing/aging_model/results

# 확인 2: MorphGAN 결과 이미지 생성 여부
!ls /content/face-aging-morphing/morphing_model/results/mom_dad/test_latest/images


ls: cannot access '/content/face-aging-morphing/aging_model/results': No such file or directory
ls: cannot access '/content/face-aging-morphing/morphing_model/results/mom_dad/test_latest/images': No such file or directory


In [11]:
# 📦 설치
!pip install gradio --quiet

import os
import subprocess
from PIL import Image, ImageDraw
import gradio as gr

# 경로 설정
AGING_DIR = "/content/face-aging-morphing/aging_model"
MORPH_DIR = "/content/face-aging-morphing/morphing_model"
DATASET_DIR = "/content/face-aging-morphing/datasets/mom_dad"
os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

# 포스터 생성 함수
def generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone):
    poster = Image.new("RGB", (600, 900), color="white")
    draw = ImageDraw.Draw(poster)
    try:
        aged_img = Image.open(aged_img_path).resize((256, 256))
        blend_img = Image.open(blended_img_path).resize((256, 256))
        poster.paste(aged_img, (50, 30))
        poster.paste(blend_img, (300, 30))
    except:
        draw.text((50, 30), "❌ 이미지 로드 실패", fill="red")

    y = 320
    spacing = 40
    draw.text((50, y), f"이름: {name}", fill="black")
    draw.text((50, y+1*spacing), f"당시 나이: {age_then}", fill="black")
    draw.text((50, y+2*spacing), f"현재 나이: {age_now}", fill="black")
    draw.text((50, y+3*spacing), f"특징: {desc}", fill="black")
    draw.text((50, y+4*spacing), f"연락처: {phone}", fill="black")
    return poster

# 전체 실행 함수
def run_pipeline(child_img, mom_img, dad_img, name, age_then, age_now, desc, phone):
    try:
        child_path = os.path.join(AGING_DIR, "input.jpg")
        mom_path = "/content/input/test1_mom.jpg"
        dad_path = "/content/input/test1_dad.jpg"
        child_img.convert("RGB").resize((256, 256)).save(child_path)
        mom_img.convert("RGB").resize((256, 256)).save(mom_path)
        dad_img.convert("RGB").resize((256, 256)).save(dad_path)

        # Fast-AgingGAN
        os.chdir(AGING_DIR)
        subprocess.run(["python", "infer.py", "--image_dir", "."], check=True)
        aged_img_path = os.path.join(AGING_DIR, "results/input_aged.jpg")
        if not os.path.exists(aged_img_path):
            return Image.new("RGB", (600, 300), color="white")

        # MorphGAN
        os.chdir(MORPH_DIR)
        subprocess.run(["cp", mom_path, f"{DATASET_DIR}/test1_A.jpg"])
        subprocess.run(["cp", dad_path, f"{DATASET_DIR}/test1_B.jpg"])
        with open(f"{DATASET_DIR}/pairs.txt", "w") as f:
            f.write("test1_A.jpg test1_B.jpg")

        subprocess.run(["chmod", "+x", "runt.sh"])
        subprocess.run(["./runt.sh", "-n", "mom_dad", "-d", "../datasets/mom_dad", "-v", "mom_dad", "-s", "256"], check=True)

        blended_img_path = f"{MORPH_DIR}/results/mom_dad/test_latest/images/test1_A_intr_1.00.png"
        if not os.path.exists(blended_img_path):
            poster = Image.new("RGB", (600, 400), color="white")
            draw = ImageDraw.Draw(poster)
            draw.text((50, 150), "❌ MorphGAN 결과 이미지가 생성되지 않았습니다.", fill="red")
            return poster

        # 포스터
        poster = generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone)
        poster.save(f"/content/output/poster_{name.replace(' ', '_')}.jpg")
        return poster

    except Exception as e:
        err = Image.new("RGB", (600, 400), color="white")
        draw = ImageDraw.Draw(err)
        draw.text((50, 150), f"❌ 실행 중 오류 발생:\n{str(e)}", fill="red")
        return err

# Gradio UI
demo = gr.Interface(
    fn=run_pipeline,
    inputs=[
        gr.Image(type="pil", label="👶 아동 사진"),
        gr.Image(type="pil", label="👩 어머니 사진"),
        gr.Image(type="pil", label="👨 아버지 사진"),
        gr.Textbox(label="1) 아동 이름"),
        gr.Textbox(label="2) 당시 나이"),
        gr.Textbox(label="3) 현재 나이"),
        gr.Textbox(label="4) 아동의 특징"),
        gr.Textbox(label="5) 연락처"),
    ],
    outputs=gr.Image(type="pil", label="🖼️ 생성된 포스터"),
    title="실종 아동 예측 시스템 (오류 표시 포함)",
    description="AI가 아동 노화와 부모 블렌딩을 통해 포스터를 자동 생성합니다."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6809c381d3730a44d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
# ✅ STEP 1: 필요한 패키지 설치
!pip install gradio pytorch-lightning
!pip install dominate

# ✅ STEP 2: 모델 레포지토리 Clone + 체크포인트 준비
!git clone https://github.com/youjung1111/face-aging-morphing.git
!mkdir -p /content/face-aging-morphing/aging_model/pretrained_model
!cp /content/converted_model.pth /content/face-aging-morphing/aging_model/pretrained_model/state_dict.pth

# ✅ STEP 3: infer.py 덮어쓰기
infer_code = """
import os
from argparse import ArgumentParser
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms
from gan_module import Generator

parser = ArgumentParser()
parser.add_argument('--image_dir', default='.', help='The image directory')

@torch.no_grad()
def main():
    args = parser.parse_args()
    image_paths = [os.path.join(args.image_dir, x) for x in os.listdir(args.image_dir)
                   if x.endswith('.png') or x.endswith('.jpg')]
    if not image_paths:
        print("❌ No input images found.")
        return

    model = Generator(ngf=32, n_residual_blocks=9)
    ckpt = torch.load('pretrained_model/state_dict.pth', map_location='cpu')
    model.load_state_dict(ckpt)
    model.eval()

    trans = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

    img = Image.open(image_paths[0]).convert('RGB')
    img_tensor = trans(img).unsqueeze(0)
    aged_face = model(img_tensor)
    aged_face = (aged_face.squeeze().permute(1, 2, 0).numpy() + 1.0) / 2.0

    os.makedirs("results", exist_ok=True)
    plt.imsave("results/input_aged.jpg", aged_face)

if __name__ == '__main__':
    main()
"""

with open("/content/face-aging-morphing/aging_model/infer.py", "w") as f:
    f.write(infer_code)

# ✅ STEP 4: Gradio 실행 함수
import os
import subprocess
from PIL import Image, ImageDraw
import gradio as gr

AGING_DIR = "/content/face-aging-morphing/aging_model"
MORPH_DIR = "/content/face-aging-morphing/morphing_model"
DATASET_DIR = "/content/face-aging-morphing/datasets/mom_dad"
os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

def generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone):
    poster = Image.new("RGB", (600, 900), color="white")
    draw = ImageDraw.Draw(poster)
    try:
        aged_img = Image.open(aged_img_path).resize((256, 256))
        blend_img = Image.open(blended_img_path).resize((256, 256))
        poster.paste(aged_img, (50, 30))
        poster.paste(blend_img, (300, 30))
    except:
        draw.text((50, 30), "❌ 이미지 로드 실패", fill="red")

    y = 320
    spacing = 40
    draw.text((50, y), f"이름: {name}", fill="black")
    draw.text((50, y+1*spacing), f"당시 나이: {age_then}", fill="black")
    draw.text((50, y+2*spacing), f"현재 나이: {age_now}", fill="black")
    draw.text((50, y+3*spacing), f"특징: {desc}", fill="black")
    draw.text((50, y+4*spacing), f"연락처: {phone}", fill="black")
    return poster

def run_pipeline(child_img, mom_img, dad_img, name, age_then, age_now, desc, phone):
    try:
        child_path = os.path.join(AGING_DIR, "input.jpg")
        mom_path = "/content/input/test1_mom.jpg"
        dad_path = "/content/input/test1_dad.jpg"
        child_img.convert("RGB").resize((256, 256)).save(child_path)
        mom_img.convert("RGB").resize((256, 256)).save(mom_path)
        dad_img.convert("RGB").resize((256, 256)).save(dad_path)

        # Fast-AgingGAN
        os.chdir(AGING_DIR)
        subprocess.run(["python", "infer.py", "--image_dir", "."], check=True)
        aged_img_path = os.path.join(AGING_DIR, "results/input_aged.jpg")
        if not os.path.exists(aged_img_path):
            return Image.new("RGB", (600, 300), color="white")

        # MorphGAN
        os.chdir(MORPH_DIR)
        subprocess.run(["cp", mom_path, f"{DATASET_DIR}/test1_A.jpg"])
        subprocess.run(["cp", dad_path, f"{DATASET_DIR}/test1_B.jpg"])
        with open(f"{DATASET_DIR}/pairs.txt", "w") as f:
            f.write("test1_A.jpg test1_B.jpg")

        subprocess.run(["chmod", "+x", "runt.sh"])
        subprocess.run(["./runt.sh", "-n", "mom_dad", "-d", "../datasets/mom_dad", "-v", "mom_dad", "-s", "256"], check=True)

        blended_img_path = f"{MORPH_DIR}/results/mom_dad/test_latest/images/test1_A_intr_1.00.png"
        if not os.path.exists(blended_img_path):
            poster = Image.new("RGB", (600, 400), color="white")
            draw = ImageDraw.Draw(poster)
            draw.text((50, 150), "❌ MorphGAN 결과 이미지가 생성되지 않았습니다.", fill="red")
            return poster

        poster = generate_poster(aged_img_path, blended_img_path, name, age_then, age_now, desc, phone)
        poster.save(f"/content/output/poster_{name.replace(' ', '_')}.jpg")
        return poster

    except Exception as e:
        err = Image.new("RGB", (600, 400), color="white")
        draw = ImageDraw.Draw(err)
        draw.text((50, 150), f"❌ 실행 중 오류 발생:\n{str(e)}", fill="red")
        return err

demo = gr.Interface(
    fn=run_pipeline,
    inputs=[
        gr.Image(type="pil", label="👶 아동 사진"),
        gr.Image(type="pil", label="👩 어머니 사진"),
        gr.Image(type="pil", label="👨 아버지 사진"),
        gr.Textbox(label="1) 아동 이름"),
        gr.Textbox(label="2) 당시 나이"),
        gr.Textbox(label="3) 현재 나이"),
        gr.Textbox(label="4) 아동의 특징"),
        gr.Textbox(label="5) 연락처"),
    ],
    outputs=gr.Image(type="pil", label="🖼️ 생성된 포스터"),
    title="실종 아동 예측 시스템 (전체 자동)",
    description="Colab 재시작에도 자동 설치, 모델 실행, 포스터 생성까지 단일 셀로 처리합니다."
)

demo.launch(share=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.6/962.6 kB 43.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-